# Build a retrieval baseline for library policies

You will build a small, fully offline RAG retrieval pipeline for a library policy assistant: index real policy text as citeable chunks, retrieve with two rankers, fuse, pack evidence into a budget, decide whether to answer or abstain, and measure evidence recall at each boundary.

There is no language model here, so nothing needs a GPU or an API key. The two retrievers are deliberately simple stand-ins: `lexical_rank` counts exact query words (in place of BM25 from C4), and `dense_rank` compares synonym-mapped word counts (in place of an embedding model). `supports` is a toy check standing in for a model or human judging whether a passage answers the question. The pipeline around them is the real thing.

Fill in each `TODO` cell, then run the check cell after it. Before submitting, restart the kernel and run all cells.

## 0. Setup (run, do not edit)

The first cell holds the corpus and the search tools. Note that `loan-policy` exists in **two revisions**: revision 2 raised the loan limit and the number of renewals. The second cell holds solutions to the lesson exercises (`chunk_tokens`, `fuse_ranks`, `pack_evidence`, `evidence_recall`), so this notebook can focus on wiring them together.

In [ ]:
import re
from collections import Counter
from math import sqrt

DOCUMENTS = [
    {"doc": "loan-policy", "rev": 1, "text": (
        "Loans. Members may borrow up to 10 items at a time. Each loan lasts 21 days. "
        "A loan may be renewed twice if no other member has placed a hold on the item. "
        "Renewals can be made online, by phone or at the front desk."
    )},
    {"doc": "loan-policy", "rev": 2, "text": (
        "Loans. Members may borrow up to 12 items at a time. Each loan lasts 21 days. "
        "A loan may be renewed three times if no other member has placed a hold on the item. "
        "Renewals can be made online, by phone or at the front desk."
    )},
    {"doc": "fines-policy", "rev": 3, "text": (
        "Late returns. Overdue items accrue a fee of 25 cents per day, capped at 5 dollars per item. "
        "Cards issued to children are exempt from overdue fees. Lost items are billed at replacement "
        "cost plus a 3 dollar processing fee. After hours, use the book drop beside the lobby doors."
    )},
    {"doc": "rooms-policy", "rev": 1, "text": (
        "Study rooms. Members can reserve a study room for up to 2 hours per day. Bookings open "
        "7 days ahead. Groups of 3 to 6 people get priority for the large rooms. Food is not "
        "allowed in study rooms, but covered drinks are fine."
    )},
]

STOPWORDS = {
    "a", "an", "the", "is", "are", "do", "does", "can", "i", "my", "me", "how", "many", "what",
    "for", "of", "in", "on", "to", "at", "per", "if", "or", "and", "be", "may", "you", "your",
    "there", "it", "with", "no", "from", "by", "up", "has", "have", "any", "but", "not", "each",
}
SYNONYMS = {
    "renewed": "renew", "renewals": "renew", "renewal": "renew", "times": "time",
    "charge": "fee", "charged": "fee", "fees": "fee", "fine": "fee", "fines": "fee", "billed": "fee",
    "daily": "day", "days": "day", "items": "item", "loans": "loan", "borrow": "loan",
    "late": "overdue", "coffee": "drink", "drinks": "drink", "sold": "sell", "sale": "sell",
    "rooms": "room", "hours": "hour",
}
LABEL_COST = 6  # tokens for the <source id="..."> wrapper around each passage


def terms(text):
    """Lower-case words minus stopwords: what the lexical search matches exactly."""
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]


def concepts(text):
    """Terms mapped through a synonym table: a toy stand-in for an embedding's sense of meaning."""
    return [SYNONYMS.get(w, w) for w in terms(text)]


def lexical_rank(query, index, limit=5):
    """Stand-in for BM25: count distinct query terms present in each chunk."""
    q = set(terms(query))
    scored = [(len(q & set(terms(c["text"]))), c["id"]) for c in index]
    return [cid for s, cid in sorted(scored, key=lambda x: (-x[0], x[1])) if s > 0][:limit]


def dense_rank(query, index, limit=5):
    """Stand-in for dense search: cosine similarity of concept-count vectors."""
    q = Counter(concepts(query))
    scored = []
    for c in index:
        d = Counter(concepts(c["text"]))
        dot = sum(q[w] * d[w] for w in q)
        if dot:
            norm = sqrt(sum(v * v for v in q.values())) * sqrt(sum(v * v for v in d.values()))
            scored.append((dot / norm, c["id"]))
    return [cid for s, cid in sorted(scored, key=lambda x: (-x[0], x[1]))][:limit]


def supports(query, text, threshold=0.75):
    """Toy support check: does the passage cover at least 75% of the query concepts?"""
    q = set(concepts(query))
    return bool(q) and len(q & set(concepts(text))) / len(q) >= threshold


def rendered_cost(chunk):
    return len(chunk["text"].split()) + LABEL_COST


def render(chunk):
    return f'<source id="{chunk["id"]}">\n{chunk["text"]}\n</source>'


In [ ]:
# Solutions to the lesson exercises, provided so this notebook can focus on integration.
def chunk_tokens(tokens, width, overlap):
    if width <= 0 or not 0 <= overlap < width:
        raise ValueError("require 0 <= overlap < width")
    chunks = []
    for start in range(0, len(tokens), width - overlap):
        end = min(start + width, len(tokens))
        chunks.append((start, end, tokens[start:end]))
        if end == len(tokens):
            break
    return chunks


def fuse_ranks(rankings, constant=60):
    if constant <= 0:
        raise ValueError("constant must be positive")
    scores = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(dict.fromkeys(ranking), 1):
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1 / (constant + rank)
    return sorted(scores, key=lambda chunk_id: (-scores[chunk_id], chunk_id))


def pack_evidence(candidates, budget):
    selected = []
    seen = set()
    for chunk_id, tokens in candidates:
        if chunk_id not in seen and tokens <= budget:
            selected.append(chunk_id)
            budget -= tokens
        seen.add(chunk_id)
    return selected


def evidence_recall(required, observed):
    gold = set(required)
    return len(gold.intersection(observed)) / len(gold) if gold else None


## 1. Stable chunk IDs

Implement `make_chunk_id(doc, rev, start, end)` and its inverse `parse_chunk_id(chunk_id)`.

- The ID format is `<doc>@r<rev>#<start>-<end>`, for example `loan-policy@r2#12-28`. `start` and `end` are word offsets, with `end` exclusive.
- Raise `ValueError` if `doc` is empty or contains `@` or `#`, if `rev` is below 1, or unless `0 <= start < end`.
- `parse_chunk_id` returns `(doc, rev, start, end)` with integers for the last three. Document IDs may contain hyphens, so split on the separators carefully.

In [ ]:
# TODO
def make_chunk_id(doc, rev, start, end):
    raise NotImplementedError

def parse_chunk_id(chunk_id):
    raise NotImplementedError


In [ ]:
assert make_chunk_id("loan-policy", 2, 12, 28) == "loan-policy@r2#12-28", "Use the format <doc>@r<rev>#<start>-<end>"
assert make_chunk_id("rooms-policy", 11, 0, 5) == "rooms-policy@r11#0-5", "Revision and offsets can have several digits"
assert parse_chunk_id("loan-policy@r2#12-28") == ("loan-policy", 2, 12, 28), "parse_chunk_id should invert make_chunk_id and return integers"
assert parse_chunk_id(make_chunk_id("a-b-c", 10, 100, 164)) == ("a-b-c", 10, 100, 164), "Hyphens in the document ID must survive a round trip"
for bad in [("", 1, 0, 1), ("x@y", 1, 0, 1), ("x#y", 1, 0, 1), ("x", 0, 0, 1), ("x", 1, 5, 5), ("x", 1, -1, 3)]:
    try:
        make_chunk_id(*bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"make_chunk_id{bad} should raise ValueError")
print("OK")


## 2. Index the current revision of each document

Implement `build_index(documents, width, overlap)`.

- Keep only the **highest revision** of each document, so a query can never see revision 1 and revision 2 of the loan policy together.
- Split each document's text into words with `text.split()`, window the words with `chunk_tokens`, and return one dict per window with keys `id`, `doc`, `rev`, `start`, `end` and `text` (the window's words joined by single spaces). Build `id` with `make_chunk_id`.
- Order the result by document ID, then by start offset, so the index is identical however `documents` is ordered.

In [ ]:
# TODO
def build_index(documents, width, overlap):
    raise NotImplementedError


In [ ]:
INDEX = build_index(DOCUMENTS, 16, 4)
ids = [c["id"] for c in INDEX]
assert len(ids) == len(set(ids)) == 12, f"Expected 12 unique chunks with width 16 and overlap 4, got {len(ids)} ({len(set(ids))} unique)"
assert all(c["rev"] == 2 for c in INDEX if c["doc"] == "loan-policy"), "Index only the latest revision of each document"
assert [c["id"] for c in build_index(DOCUMENTS[::-1], 16, 4)] == ids, "The index must not depend on the order of the input documents"
words = {(d["doc"], d["rev"]): d["text"].split() for d in DOCUMENTS}
for c in INDEX:
    assert parse_chunk_id(c["id"]) == (c["doc"], c["rev"], c["start"], c["end"]), f"{c['id']} does not match its doc, rev and offset fields"
    assert " ".join(words[(c["doc"], c["rev"])][c["start"]:c["end"]]) == c["text"], f"The offsets of {c['id']} do not point at its text"
assert "loan-policy@r2#12-28" in ids, "Expected a chunk loan-policy@r2#12-28: check the ID format and the window arithmetic"
print(len(INDEX), "chunks, for example:", INDEX[5]["id"], "->", INDEX[5]["text"])
print("OK")


## 3. Trace evidence through fusion and packing

Implement `evidence_report(rankings, costs, budget, required)`, where `costs` maps chunk IDs to rendered token costs.

1. Fuse `rankings` with `fuse_ranks` and constant 60.
2. Keep only the fused IDs that have an entry in `costs`, in fused order. These are the **candidates**.
3. Pack the candidates with `pack_evidence`, using their costs and the budget.
4. Return `(packed_ids, candidate_recall, packed_recall)`, where candidate recall is `evidence_recall` over the candidates (fused IDs that have a cost entry) and packed recall is `evidence_recall` over the packed IDs. Both are `None` when `required` is empty.

In [ ]:
# TODO
def evidence_report(rankings, costs, budget, required):
    raise NotImplementedError


In [ ]:
assert evidence_report([["hours", "exceptions"], ["exceptions", "parking"]], {"hours": 2, "exceptions": 4, "parking": 2}, 5, ["hours", "exceptions"]) == (["exceptions"], 1.0, 0.5), "Fuse first: exceptions is in both lists, so it is packed first and fills most of the budget"
assert evidence_report([["unknown", "a"]], {"a": 1}, 1, ["unknown", "a"]) == (["a"], 0.5, 0.5), "Candidate recall counts only fused IDs that have a cost entry"
assert evidence_report([["b", "a"], ["a"]], {"a": 3, "b": 3}, 3, ["b"]) == (["a"], 1.0, 0.0), "A required chunk can be retrieved but lost to the budget"
assert evidence_report([[]], {}, 4, []) == ([], None, None), "With no required evidence both recalls are None"
print("OK")


## 4. Run the pipeline end to end

Implement `run_query(query, index, budget, required=())`. It should:

1. rank the index with both `lexical_rank` and `dense_rank`;
2. build a `costs` dict of `rendered_cost(chunk)` for every chunk in the index;
3. call your `evidence_report` with the two rankings;
4. decide `"answer"` if at least one packed chunk `supports` the query, otherwise `"abstain"`;
5. return a dict with keys `query`, `lexical`, `dense`, `packed`, `candidate_recall`, `packed_recall`, `decision` and `evidence` (the packed chunks rendered with `render`, joined by newlines; this is the block a model would receive).

The next cell runs a fixed query set with human annotations: two answerable questions and one the policies do not answer.

In [ ]:
# TODO
def run_query(query, index, budget, required=()):
    raise NotImplementedError


In [ ]:
BUDGET = 50
QUERIES = [
    ("How many times can I renew a loan?", ["loan-policy@r2#12-28"]),
    ("What is the daily charge for late items?", ["fines-policy@r3#0-16"]),
    ("Is coffee sold in the lobby?", []),  # not covered by any policy: the right outcome is to abstain
]
RESULTS = [run_query(q, INDEX, BUDGET, required) for q, required in QUERIES]
for r in RESULTS:
    print(r["query"])
    print("  lexical:", r["lexical"])
    print("  dense:  ", r["dense"])
    print("  packed: ", r["packed"])
    print("  recall (candidates, packed):", r["candidate_recall"], r["packed_recall"], "| decision:", r["decision"])
print()
print(RESULTS[0]["evidence"])


In [ ]:
renew, fee, coffee = RESULTS
costs = {c["id"]: rendered_cost(c) for c in INDEX}
for r in RESULTS:
    assert sum(costs[cid] for cid in r["packed"]) <= BUDGET, f"Packed evidence for {r['query']!r} exceeds the budget"
    assert r["evidence"].count("<source id=") == len(r["packed"]), "Render every packed chunk, and only those, into the evidence block"
assert renew["decision"] == "answer" and renew["packed_recall"] == 1.0, "The renewal question should be answered from loan-policy@r2#12-28"
assert not any("@r1#" in cid for cid in renew["lexical"] + renew["dense"] + renew["packed"] if cid.startswith("loan-policy")), "Revision 1 of the loan policy must not be retrievable"
assert fee["decision"] == "answer" and fee["candidate_recall"] == 1.0, "The fee question should retrieve and answer from fines-policy@r3#0-16"
assert coffee["decision"] == "abstain", "No policy covers coffee sales: the pipeline must abstain even though some chunks matched a word"
assert coffee["packed"] and coffee["candidate_recall"] is None, "The coffee query retrieves loosely related chunks, and has no required evidence"
tight = run_query("How many times can I renew a loan?", INDEX, 20, ["loan-policy@r2#12-28"])
assert (tight["candidate_recall"], tight["packed_recall"], tight["decision"]) == (1.0, 0.0, "abstain"), "With a 20-token budget the right chunk is retrieved but cannot be packed"
print("OK")


## 5. Interpretation

Answer in this cell, using the printed output above.

1. For the renewal question and the coffee question, record the query, the lexical and dense IDs, the packed IDs and the decision. For the renewal question, quote the passage that supports the answer and give the answer with its citation.
2. The coffee question retrieved chunks, yet the pipeline abstained. Which chunks matched, and why? What would a real system need instead of the toy `supports` check?
3. The tight-budget run had candidate recall 1.0 and packed recall 0.0. Which stage would you fix, and how?
4. Suppose a user reports that the assistant said "two renewals". Using the chunk IDs, explain how you would tell whether the cause was indexing (a stale revision), retrieval, packing or answer generation.

**Your answer:**